In [1]:

import argparse
import logging
from types import SimpleNamespace
import os
from pathlib import Path
import csv
import sys
import math
import numpy as np
import pandas as pd
import time
import joblib
import json 
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
# sys.path.append('/Users/yiqin/Documents/PythonProjects/GraspDataProcessing/src')
# sys.path.append('D:\\PythonPrograms\\GraspDataProcessing\\src')
sys.path.append('D:\\PythonProjects\\GraspDataProcessing\\src')
import graspdataprocessing as gdp


In [2]:
config_file = 'config.toml'  # 配置文件的路径
config = gdp.load_config(config_file)

配置验证通过: cutoff_value=1e-09, initial_ratio=0.09


In [3]:
"""主程序逻辑"""
config.file_name = f'{config.conf}_{config.cal_loop_num}'
logger = gdp.setup_logging(config)
logger.info("机器学习训练程序启动")
execution_time = time.time()

gdp.setup_directories(config)
# 初始化结果文件
gdp.initialize_results_file(config, logger)

# 验证初始文件
gdp.validate_initial_files(config, logger)

logger.info(f"初始比例: {config.initial_ratio}")
logger.info(f"光谱项: {config.spetral_term}")

2025-06-11 09:46:55,537 - INFO - 机器学习训练程序启动
2025-06-11 09:46:55,538 - INFO - 成功加载初始CSFs文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4.c
2025-06-11 09:46:55,539 - INFO - 初始比例: 0.09
2025-06-11 09:46:55,539 - INFO - 光谱项: ['5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_9D', '5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_7D']


In [4]:
try:
    # 加载数据文件
    energy_level_data_pd, rmix_file_data, target_pool_csfs_data, raw_csfs_descriptors, cal_csfs_data, caled_csfs_indices_dict = gdp.load_data_files(config, logger)
    
    # 检查组态耦合
    cal_result = gdp.check_configuration_coupling(config, energy_level_data_pd, logger)
    logger.info("************************************************")

except Exception as e:
        logger.error(f"程序执行过程中发生错误: {str(e)}")
        raise

2025-06-11 09:46:55,548 - INFO - 加载能级数据: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level


Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded.
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded.
file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded
data file type: level data
energy levels data has been written into LEVEL/cv4odd1as3_odd4_1.level_level.csv csv file
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.m loaded.
data file type: rmcdhf mix_coefficient_data
g92mix: G92MIX
 nblock = 1,       ncftot =   385600,          nw =  41,            nelec =   64


100%|██████████| 1/1 [00:00<00:00, 999.60it/s]
2025-06-11 09:46:55,553 - INFO - 加载 *.m 文件数据: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.m


cycle jblock = 1
 Block no. = 1, 2J+1 = 9, Parity = -1, No. of eigenvalues = 2, No. of CSFs = 385600

    Energy levels for ...
Rydberg constant is  109737.31568508

---------------------------------------------
 No Pos  J  Parity   Energy Total    Levels
                      (a.u.)         (cm^-1)
---------------------------------------------

  1  1   4   +    -11274.7413433        0.00
  2  0   4   +    -11274.7204780     4579.40


2025-06-11 09:47:01,617 - INFO - 加载初始 CSFs 文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4
2025-06-11 09:47:01,907 - INFO - 加载初始 CSFs 描述符文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4


Descriptors loaded from: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_descriptors.npy
Labels loaded from: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_descriptors_block_indices.npy
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.c loaded.


2025-06-11 09:47:03,114 - INFO - 加载本轮计算 CSFs 文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.c
2025-06-11 09:47:03,126 - INFO - 加载本轮选择的 CSFs 的索引文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1_chosen_indices.pkl
2025-06-11 09:47:03,128 - INFO - cal_loop 1 组态耦合正确
2025-06-11 09:47:03,129 - INFO - ************************************************


In [5]:
caled_csfs_descriptors = gdp.generate_chosen_csfs_descriptors(config, caled_csfs_indices_dict, raw_csfs_descriptors, rmix_file_data, logger)

2025-06-11 09:47:03,292 - INFO - 保存本轮选择的 CSFs 的描述符文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1/cv4odd1as3_odd4_1.npy


Descriptors saved to: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1_descriptors.npy


In [6]:
stay_csfs_descriptors = gdp.get_stay_descriptors(raw_csfs_descriptors, caled_csfs_indices_dict)

In [7]:
X_stay = stay_csfs_descriptors.copy()

In [8]:
model, X_train, X_test, y_train, y_test, training_time, weight = gdp.train_model(config, caled_csfs_descriptors, rmix_file_data, logger)

2025-06-11 09:47:04,118 - INFO - 训练集 - 正样本:22038, 负样本:286442, 比例:0.0714
2025-06-11 09:47:05,044 - INFO - 创建新模型，设置类别权重: 负样本=1.0, 正样本=13.0
2025-06-11 09:47:05,045 - INFO - 使用原始数据训练 - 不进行重采样
2025-06-11 09:47:05,045 - INFO - 最终训练数据 - 正样本:22038, 负样本:286442, 比例:0.0714
2025-06-11 09:47:05,046 - INFO - 使用类别权重和损失函数来处理数据不平衡问题
2025-06-11 09:47:05,046 - INFO -              训练模型
2025-06-11 09:47:05,047 - INFO - 开始训练 - 数据量:308,480, 特征维度:87
2025-06-11 09:47:20,300 - INFO - Epoch [50/150], Loss: 0.327611
2025-06-11 09:47:35,528 - INFO - Epoch [100/150], Loss: 0.314877
2025-06-11 09:47:50,773 - INFO - Epoch [150/150], Loss: 0.307532
2025-06-11 09:47:50,774 - INFO - 训练Loss良好 (0.310)，模型收敛效果理想
2025-06-11 09:47:50,774 - INFO -              预测与评估
2025-06-11 09:47:50,863 - INFO - 预测概率统计 - 最小值:0.0015, 最大值:1.0000, 平均值:0.2593
2025-06-11 09:47:50,864 - INFO - 预测为正类的样本数: 5568/77120
2025-06-11 09:47:50,864 - INFO - 真实正样本数: 5423.0/77120
2025-06-11 09:47:50,865 - INFO - 真实正样本比例: 0.070, 预测正样本比例: 0.072


(385600,)


2025-06-11 09:47:52,356 - INFO - 测试集预测结果:
2025-06-11 09:47:52,356 - INFO - AUC:0.8765547979048046, pr_auc:0.7693645649386659, f1:0.7533436448002911, accuracy:0.9648469917012448, precision:0.7435344827586207, recall:0.7634150839018993
2025-06-11 09:47:52,455 - INFO - 训练集预测结果:
2025-06-11 09:47:52,456 - INFO - AUC:0.9457053534321376, f1:0.7698690054787726, accuracy:0.9669119553941908, precision:0.7650907461348868, recall:0.7747073237135856


In [12]:
X_test

array([[1., 1., 1., ..., 0., 0., 8.],
       [2., 0., 0., ..., 0., 0., 8.],
       [1., 1., 1., ..., 0., 0., 8.],
       ...,
       [2., 0., 0., ..., 0., 0., 8.],
       [2., 0., 0., ..., 0., 0., 8.],
       [1., 1., 1., ..., 0., 0., 8.]], dtype=float32)

In [13]:
evaluation_results = gdp.evaluate_model(
            model, X_train, X_test, y_train, y_test, X_stay, config, logger
        )

2025-06-11 09:50:52,404 - INFO - 开始预测与评估
2025-06-11 09:50:54,397 - INFO - 预测结果与模型保存成功


In [11]:
evaluation_results

{'y_pred_other': array([False, False, False, ..., False, False, False]),
 'eval_time': 0.44977259635925293,
 'metrics': {'f1': 0.7533436448002911,
  'roc_auc': np.float64(0.8765547979048046),
  'accuracy': 0.9648469917012448,
  'precision': 0.7435344827586207,
  'recall': 0.7634150839018993,
  'f1_train': 0.7698690054787726,
  'roc_auc_train': np.float64(0.9457053534321376),
  'accuracy_train': 0.9669119553941908,
  'precision_train': 0.7650907461348868,
  'recall_train': 0.7747073237135856}}